In [6]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [7]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}
REVIEWS = {
    "wireless headphones": {"reviews": 1162, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)
    
@tool
def get_product(name:str)->str:
    """Look up a product by name and return its price, rating , stock and description"""
    p=PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)


In [8]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0,
)


agent = create_agent(
    llm,
    tools=[get_product,get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [9]:
def ask(question:str):
    result = agent.invoke({"messages":[{"role":"user","content":question}]})
    print(result["messages"][-1].content)

In [10]:
from langgraph.checkpoint.memory import InMemorySaver
agent2 = create_agent(
    llm,
    tools=[get_product,get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer = InMemorySaver()
)
def ask2(question:str):
    config = {"configurable":{"thread_id":"user-alice-session-1"}}
    result = agent2.invoke(
        {"messages":[{"role":"user","content":question}]},
        config
    )
    print(result["messages"][-1].content)

In [11]:
ask2("what is the price of wireless headphones")

[{'type': 'text', 'text': 'The price of the wireless headphones is **$79.99**. \n\n**Product Highlights:**\n* Over-ear Bluetooth design\n* 30-hour battery life\n* Active noise cancellation', 'extras': {'signature': 'EssCCsgCAWkUfRPs7PQQqalPbg8+R4rR5p5yyhNmJYi7c2lEhRd+kLI0soMx66HUwNqTxfe1vgD5uL3kxVVZ9dczr9+TT0pbZxa1Vz80iF/hvroaRxjCnLrl/R/CvEp3HkT91uuiSbQujby5hgxNk4TXDhLj5sbJ9xg6ktI1C0CJ/fPbBt7GO09d1RIVmcVWjMz+teJG3EaqFmG/CjdqsmKOywIU3fsfjA+l7cYmYuC/fXJf5el2eKFUFLT0VaNHMcbAub2h9svWydQ/0n2ceAraNOTRbPFr0Qab8Qy7VXhFnqmshQSUJbP8rsO6R/iqoyXursYuOMMgOZZPwyQuDgyY/YigaCJRDAVjmZHko4iIRUZFJMzbNJc2rmJf2jUl9svn4ZAUlSYMIViE/9bzbrO70RfIMaz+wvg0qeIhUPCyhE+RMfYzPNYrHYI/4w=='}}]


In [12]:
ask2("what are the reviews on this product")

[{'type': 'text', 'text': 'The wireless headphones have a rating of **4.6 out of 5 stars** based on **1,162 reviews**.', 'extras': {'signature': 'Et4BCtsBAWkUfRM43kD/Jl3T2SzjcERzhySwmn3dzdybWwYUHZpcLbWkSawhdpVhi5CNGP134P39055tRSHPMQMplZ2/jel5WGBmcv4Sel9+f324BjCYUVFolg6E8N5Al7UkZqYITQB0ijEZhOR24fNG1r+kYL/wJ/8e7+YcPtIsWbK683AO7l+CCTo0jU66OZoXCjHWFTPlk0QaOvJpm1ZZkAWKIqZviDDL9NLiPjsQyAZmWPc/Co3tTN/gPS6iwjEHAf2pjr4hnlu8i4qDj4y6wtzO9FGChi1idSKd7IbW'}}]
